In [14]:
# ============================================================
# WEEK 2: PYTHON FOR DATA ANALYSIS & VISUALIZATION
# LECTURE 2: WORLD HAPPINESS REPORT 2023
# CELL 1: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [15]:
# ============================================================
# WEEK 2: PYTHON FOR DATA ANALYSIS & VISUALIZATION
# LECTURE 2: WORLD HAPPINESS REPORT 2023
# CELL 2: LOAD AND PREPARE THE DATASET
# ============================================================

# ------------------------------------------------------------
# STEP 1: LOCATE AND LOAD THE DATASET
# ------------------------------------------------------------

possible_paths = [
    Path("/kaggle/working/WHR2023.csv"),
    Path("WHR2023.csv"),
    Path("../data/WHR2023.csv"),
]

kaggle_input = Path("/kaggle/input")

if kaggle_input.exists():
    possible_paths.extend(kaggle_input.rglob("WHR2023.csv"))

csv_path = next(
    (path for path in possible_paths if path.is_file()),
    None
)

if csv_path is None:
    raise FileNotFoundError(
        "WHR2023.csv was not found. Please check the dataset path."
    )

df = pd.read_csv(csv_path)

print(f"Dataset loaded successfully: {csv_path}")
print(f"Original dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")


# ------------------------------------------------------------
# STEP 2: STANDARDIZE COLUMN NAMES
# ------------------------------------------------------------

df.columns = df.columns.str.strip()

column_mapping = {
    "Country name": "Country",
    "Ladder score": "Happiness_Score",
    "Logged GDP per capita": "GDP",
    "Social support": "Social_Support",
    "Healthy life expectancy": "Life_Expectancy",
    "Freedom to make life choices": "Freedom",
    "Generosity": "Generosity",
    "Perceptions of corruption": "Corruption",
}

df = df.rename(columns=column_mapping)

required_columns = [
    "Country",
    "Happiness_Score",
    "GDP",
    "Freedom",
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}"
    )


# ------------------------------------------------------------
# STEP 3: CLEAN AND VALIDATE THE DATA
# ------------------------------------------------------------

numeric_columns = [
    "Happiness_Score",
    "GDP",
    "Freedom",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df["Country"] = (
    df["Country"]
      .astype("string")
      .str.strip()
)

df = df.dropna(
    subset=["Country", "Happiness_Score"]
).copy()

if df["Country"].duplicated().any():
    duplicates = (
        df.loc[df["Country"].duplicated(keep=False), "Country"]
          .unique()
          .tolist()
    )

    raise ValueError(
        f"Duplicate country records found: {duplicates}"
    )


# ------------------------------------------------------------
# STEP 4: MAP COUNTRIES TO REGIONS
# ------------------------------------------------------------
# WHR2023.csv does not contain a Region column.
# These broad geographical groupings are added for this assignment.
# They are not official regional classifications from the CSV.

region_countries = {

    "Western Europe": [
        "Andorra", "Austria", "Belgium", "Cyprus", "Denmark",
        "Finland", "France", "Germany", "Greece", "Iceland",
        "Ireland", "Italy", "Luxembourg", "Malta", "Netherlands",
        "Norway", "Portugal", "Spain", "Sweden", "Switzerland",
        "United Kingdom",
    ],

    "Latin America": [
        "Argentina", "Bolivia", "Brazil", "Chile", "Colombia",
        "Costa Rica", "Dominican Republic", "Ecuador",
        "El Salvador", "Guatemala", "Haiti", "Honduras",
        "Jamaica", "Mexico", "Nicaragua", "Panama", "Paraguay",
        "Peru", "Uruguay", "Venezuela",
    ],

    "East Asia": [
        "China", "Hong Kong S.A.R. of China", "Japan",
        "Mongolia", "North Korea", "South Korea",
        "Taiwan Province of China",
    ],

    "Sub-Saharan Africa": [
        "Angola", "Benin", "Botswana", "Burkina Faso", "Burundi",
        "Cameroon", "Central African Republic", "Chad", "Comoros",
        "Congo", "Congo (Brazzaville)", "Congo (Kinshasa)",
        "Cote d'Ivoire", "Ivory Coast", "Djibouti",
        "Equatorial Guinea", "Eritrea", "Eswatini", "Swaziland",
        "Ethiopia", "Gabon", "Gambia", "Ghana", "Guinea",
        "Guinea-Bissau", "Kenya", "Lesotho", "Liberia",
        "Madagascar", "Malawi", "Mali", "Mauritania", "Mauritius",
        "Mozambique", "Namibia", "Niger", "Nigeria", "Rwanda",
        "Senegal", "Sierra Leone", "Somalia", "South Africa",
        "South Sudan", "Sudan", "Tanzania", "Togo", "Uganda",
        "Zambia", "Zimbabwe",
    ],

    "South Asia": [
        "Afghanistan", "Bangladesh", "Bhutan", "India",
        "Maldives", "Nepal", "Pakistan", "Sri Lanka",
    ],

    "Southeast Asia": [
        "Brunei", "Cambodia", "Indonesia", "Laos", "Malaysia",
        "Myanmar", "Philippines", "Singapore", "Thailand",
        "Timor-Leste", "Vietnam",
    ],

    "Central and Eastern Europe": [
        "Albania", "Armenia", "Azerbaijan", "Belarus",
        "Bosnia and Herzegovina", "Bulgaria", "Croatia",
        "Czechia", "Estonia", "Georgia", "Hungary", "Kazakhstan",
        "Kosovo", "Kyrgyzstan", "Latvia", "Lithuania", "Moldova",
        "Montenegro", "North Macedonia", "Poland", "Romania",
        "Russia", "Serbia", "Slovakia", "Slovenia", "Tajikistan",
        "Turkmenistan", "Ukraine", "Uzbekistan",
    ],

    "Middle East and North Africa": [
        "Algeria", "Bahrain", "Egypt", "Iran", "Iraq", "Israel",
        "Jordan", "Kuwait", "Lebanon", "Libya", "Morocco", "Oman",
        "Palestinian Territories", "State of Palestine",
        "Qatar", "Saudi Arabia", "Syria", "Tunisia", "Turkey",
        "Türkiye", "Turkiye", "United Arab Emirates", "Yemen",
    ],

    "North America and ANZ": [
        "Australia", "Canada", "New Zealand", "United States",
    ],
}


# ------------------------------------------------------------
# STEP 5: CREATE A COUNTRY-TO-REGION LOOKUP
# ------------------------------------------------------------

country_to_region = {
    country: region
    for region, countries in region_countries.items()
    for country in countries
}

# Normalize names to handle common spelling and naming variations.
# For example, Turkiye and Türkiye are treated as equivalent.

def normalize_country_name(country):
    country = str(country).strip().casefold()

    aliases = {
        "turkiye": "turkey",
        "state of palestine": "palestinian territories",
    }

    return aliases.get(country, country)


normalized_country_to_region = {
    normalize_country_name(country): region
    for country, region in country_to_region.items()
}

df["Region"] = df["Country"].map(
    lambda country: normalized_country_to_region.get(
        normalize_country_name(country)
    )
)


# ------------------------------------------------------------
# STEP 6: CHECK FOR UNMAPPED COUNTRIES
# ------------------------------------------------------------

unmapped_countries = sorted(
    df.loc[df["Region"].isna(), "Country"]
      .unique()
      .tolist()
)

if unmapped_countries:
    print("\nWARNING: The following countries are not mapped:")

    for country in unmapped_countries:
        print(f" - {country}")

    raise ValueError(
        "Some countries still lack regional classifications. "
        "Add them to region_countries before running Tasks 1 and 3."
    )


# ------------------------------------------------------------
# STEP 7: VALIDATE AND DISPLAY THE PREPARED DATA
# ------------------------------------------------------------

print("\nDataset preparation completed successfully!")
print(f"Countries available: {len(df)}")
print(f"Regions represented: {df['Region'].nunique()}")

print("\nCountries per region:")

display(
    df["Region"]
      .value_counts()
      .rename_axis("Region")
      .reset_index(name="Number of Countries")
)

print("\nMissing values in the analysis columns:")

display(
    df[
        [
            "Country",
            "Region",
            "Happiness_Score",
            "GDP",
            "Freedom",
        ]
    ]
    .isna()
    .sum()
    .rename("Missing Values")
    .to_frame()
)

print("\nPrepared dataset preview:")

display(
    df[
        [
            "Country",
            "Region",
            "Happiness_Score",
            "GDP",
            "Freedom",
        ]
    ].head(10)
)

Dataset loaded successfully: /kaggle/input/datasets/ajaypalsinghlo/world-happiness-report-2023/WHR2023.csv
Original dataset shape: 137 rows, 19 columns

Dataset preparation completed successfully!
Countries available: 137
Regions represented: 9

Countries per region:


,Region,Number of Countries
0,Sub-Saharan Africa,33
1,Central and Eastern Europe,26
2,Western Europe,20
3,Latin America,19
4,Middle East and North Africa,14
5,Southeast Asia,9
6,East Asia,6
7,South Asia,6
8,North America and ANZ,4



Missing values in the analysis columns:


,Missing Values
Country,0
Region,0
Happiness_Score,0
GDP,0
Freedom,0



Prepared dataset preview:


,Country,Region,Happiness_Score,GDP,Freedom
0,Finland,Western Europe,7.804,10.792,0.961
1,Denmark,Western Europe,7.586,10.962,0.934
2,Iceland,Western Europe,7.530,10.896,0.936
3,Israel,Middle East and North Africa,7.473,10.639,0.809
4,Netherlands,Western Europe,7.403,10.942,0.887
5,Sweden,Western Europe,7.395,10.883,0.948
6,Norway,Western Europe,7.315,11.088,0.947
7,Switzerland,Western Europe,7.240,11.164,0.891
8,Luxembourg,Western Europe,7.228,11.660,0.915
9,New Zealand,North America and ANZ,7.123,10.662,0.887


In [16]:
# ============================================================
# TASK 1: AVERAGE HAPPINESS SCORE BY REGION
# ============================================================

# Calculate the average happiness score for each region
region_avg = (
    df.groupby("Region", as_index=False)
      .agg(Happiness_Score=("Happiness_Score", "mean"))
      .sort_values("Happiness_Score", ascending=False)
      .reset_index(drop=True)
)

if region_avg.empty:
    raise ValueError("No regional data is available for this chart.")

if region_avg["Happiness_Score"].isna().any():
    raise ValueError("Some regional happiness averages are missing.")

# Identify the highest-scoring region
top_region = region_avg.iloc[0]

# Highlight the leading region
bar_colours = [
    "#2878B5" if region == top_region["Region"] else "#CBD5E1"
    for region in region_avg["Region"]
]

# Display the results
print("Average Happiness Score by Region")

display(
    region_avg.style
        .format({"Happiness_Score": "{:.2f}"})
        .hide(axis="index")
)

# Create the horizontal bar chart
fig1 = go.Figure()

fig1.add_trace(
    go.Bar(
        x=region_avg["Happiness_Score"],
        y=region_avg["Region"],
        orientation="h",
        marker=dict(color=bar_colours),
        text=region_avg["Happiness_Score"].map(
            lambda value: f"{value:.2f}"
        ),
        textposition="outside",
        textfont=dict(size=12, color="#263238"),
        cliponaxis=False,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Average Happiness Score: %{x:.2f}"
            "<extra></extra>"
        )
    )
)

# Professional chart layout
fig1.update_layout(
    title=dict(
        text=(
            f"{top_region['Region']} Leads in Average Happiness"
            "<br><sup>"
            f"Regional average: {top_region['Happiness_Score']:.2f} "
            "| World Happiness Report 2023"
            "</sup>"
        ),
        x=0.02,
        xanchor="left",
        font=dict(size=21, color="#1F2933")
    ),
    xaxis=dict(
        title="Average Happiness Score",
        range=[
            0,
            region_avg["Happiness_Score"].max() * 1.20
        ],
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor="#52616B",
        showgrid=True,
        gridcolor="#E9EEF2",
        tickfont=dict(size=11),
        automargin=True
    ),
    yaxis=dict(
        title="",
        autorange="reversed",
        showgrid=False,
        ticks="",
        tickfont=dict(size=12),
        automargin=True
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(
        family="Arial, sans-serif",
        color="#263238"
    ),
    margin=dict(l=230, r=100, t=125, b=75),
    height=max(550, len(region_avg) * 48 + 180),
    showlegend=False,
    bargap=0.30,
    hoverlabel=dict(
        bgcolor="white",
        font_size=12
    )
)

# Display the chart underneath this cell
fig1.show()

Average Happiness Score by Region


Region,Happiness_Score
North America and ANZ,7.02
Western Europe,6.89
Latin America,5.97
Central and Eastern Europe,5.95
East Asia,5.93
Southeast Asia,5.43
Middle East and North Africa,5.10
Sub-Saharan Africa,4.38
South Asia,4.09


In [17]:
# ============================================================
# TASK 2: TOP 8 VS. BOTTOM 8 HAPPIEST COUNTRIES
# ============================================================

# Check that the required data is available
if df["Happiness_Score"].isna().any():
    raise ValueError("Missing happiness scores detected.")

if df["Country"].isna().any():
    raise ValueError("Missing country names detected.")

if len(df) < 16:
    raise ValueError("At least 16 countries are needed for this comparison.")

# Select the eight highest-scoring countries
top8 = (
    df.nlargest(8, "Happiness_Score")
      [["Country", "Happiness_Score"]]
      .copy()
)

top8["Group"] = "Top 8 Happiest"

# Select the eight lowest-scoring countries
bottom8 = (
    df.nsmallest(8, "Happiness_Score")
      [["Country", "Happiness_Score"]]
      .copy()
)

bottom8["Group"] = "Bottom 8 Happiest"

# Combine the two groups
combined = pd.concat(
    [bottom8, top8],
    ignore_index=True
)

# Sort from lowest to highest score
combined = (
    combined.sort_values("Happiness_Score", ascending=True)
            .reset_index(drop=True)
)

# Calculate overall statistics from the complete dataset
global_avg = df["Happiness_Score"].mean()
highest_score = df["Happiness_Score"].max()
lowest_score = df["Happiness_Score"].min()
score_gap = highest_score - lowest_score

highest_country = df.loc[
    df["Happiness_Score"].idxmax(), "Country"
]

lowest_country = df.loc[
    df["Happiness_Score"].idxmin(), "Country"
]

# Display summary statistics
print(f"Global Average Happiness Score: {global_avg:.2f}")
print(f"Highest-Scoring Country: {highest_country} ({highest_score:.2f})")
print(f"Lowest-Scoring Country: {lowest_country} ({lowest_score:.2f})")
print(f"Difference Between Highest and Lowest: {score_gap:.2f} points")

print("\nSelected Countries")

display(
    combined[["Country", "Group", "Happiness_Score"]]
        .style
        .format({"Happiness_Score": "{:.2f}"})
        .hide(axis="index")
)

# Define a restrained colour palette
colour_map = {
    "Top 8 Happiest": "#2878B5",
    "Bottom 8 Happiest": "#E67E22"
}

# Create the horizontal bar chart
fig2 = go.Figure()

for group_name in ["Bottom 8 Happiest", "Top 8 Happiest"]:

    group_data = combined[
        combined["Group"] == group_name
    ]

    fig2.add_trace(
        go.Bar(
            x=group_data["Happiness_Score"],
            y=group_data["Country"],
            orientation="h",
            name=group_name,
            marker=dict(color=colour_map[group_name]),
            text=group_data["Happiness_Score"].map(
                lambda value: f"{value:.2f}"
            ),
            textposition="outside",
            textfont=dict(size=11, color="#263238"),
            cliponaxis=False,
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Happiness Score: %{x:.2f}"
                "<extra></extra>"
            )
        )
    )

# Add a reference line for the global average
fig2.add_vline(
    x=global_avg,
    line_dash="dash",
    line_color="#374151",
    line_width=2,
    annotation_text=f"Global Average: {global_avg:.2f}",
    annotation_position="top",
    annotation_font=dict(size=11, color="#374151")
)

# Add a score-gap annotation near the highest-scoring country
fig2.add_annotation(
    x=highest_score,
    y=highest_country,
    text=f"Score gap: {score_gap:.2f} points",
    showarrow=True,
    arrowhead=2,
    arrowwidth=1.5,
    arrowcolor="#374151",
    ax=-95,
    ay=-42,
    bgcolor="white",
    bordercolor="#CBD5E1",
    borderwidth=1,
    borderpad=6,
    font=dict(size=11, color="#263238")
)

# Professional chart layout
fig2.update_layout(
    title=dict(
        text=(
            "A Wide Happiness Gap Separates the Highest- and Lowest-Scoring Countries"
            "<br><sup>"
            "Eight highest- and eight lowest-scoring countries "
            "| World Happiness Report 2023"
            "</sup>"
        ),
        x=0.02,
        xanchor="left",
        font=dict(size=20, color="#1F2933")
    ),
    xaxis=dict(
        title="Happiness Score",
        range=[0, highest_score * 1.28],
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor="#52616B",
        showgrid=True,
        gridcolor="#E9EEF2",
        tickfont=dict(size=11),
        automargin=True
    ),
    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=combined["Country"].tolist(),
        showgrid=False,
        ticks="",
        tickfont=dict(size=11),
        automargin=True
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(
        family="Arial, sans-serif",
        color="#263238"
    ),
    margin=dict(l=180, r=145, t=150, b=80),
    height=800,
    bargap=0.28,
    barmode="overlay",
    legend=dict(
        title="Country Group",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    hoverlabel=dict(
        bgcolor="white",
        font_size=12
    )
)

# Display the chart underneath this cell
fig2.show()

Global Average Happiness Score: 5.54
Highest-Scoring Country: Finland (7.80)
Lowest-Scoring Country: Afghanistan (1.86)
Difference Between Highest and Lowest: 5.95 points

Selected Countries


Country,Group,Happiness_Score
Afghanistan,Bottom 8 Happiest,1.86
Lebanon,Bottom 8 Happiest,2.39
Sierra Leone,Bottom 8 Happiest,3.14
Zimbabwe,Bottom 8 Happiest,3.20
Congo (Kinshasa),Bottom 8 Happiest,3.21
Botswana,Bottom 8 Happiest,3.44
Malawi,Bottom 8 Happiest,3.50
Comoros,Bottom 8 Happiest,3.54
Switzerland,Top 8 Happiest,7.24
Norway,Top 8 Happiest,7.32


In [18]:
# ============================================================
# TASK 3: GDP VS. FREEDOM ACROSS FIVE REGIONS
# ============================================================

# Regions specified in the assignment
selected_regions = [
    "Western Europe",
    "Latin America",
    "East Asia",
    "Sub-Saharan Africa",
    "South Asia"
]

# Validate the required columns
required_columns = ["Region", "GDP", "Freedom"]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

# Select countries belonging to the five requested regions
regional_df = df[
    df["Region"].isin(selected_regions)
].copy()

# Check that every requested region is represented
available_regions = set(
    regional_df["Region"].dropna()
)

missing_regions = [
    region for region in selected_regions
    if region not in available_regions
]

if missing_regions:
    raise ValueError(
        f"No matching countries were found for: {missing_regions}. "
        "Check the region mapping in Cell 2."
    )

# Calculate the mean GDP and freedom indicators for each region
regional_avg = (
    regional_df.groupby("Region")[["GDP", "Freedom"]]
               .mean()
               .reindex(selected_regions)
               .reset_index()
)

# Exclude missing values from the calculation, but stop if an entire
# region lacks data for either indicator
if regional_avg[["GDP", "Freedom"]].isna().any().any():
    raise ValueError(
        "At least one region has a missing GDP or Freedom average. "
        "Check the source data."
    )

# Display the results
print("Average GDP and Freedom Indicators by Region")

display(
    regional_avg.style
        .format({
            "GDP": "{:.2f}",
            "Freedom": "{:.2f}"
        })
        .hide(axis="index")
)

# Reshape the data for grouped bars
regional_long = regional_avg.melt(
    id_vars="Region",
    value_vars=["GDP", "Freedom"],
    var_name="Indicator",
    value_name="Average Value"
)

# Define two contrasting colours
indicator_colours = {
    "GDP": "#2878B5",
    "Freedom": "#E67E22"
}

# Create the grouped bar chart
fig3 = px.bar(
    regional_long,
    x="Region",
    y="Average Value",
    color="Indicator",
    barmode="group",
    text=regional_long["Average Value"].map(
        lambda value: f"{value:.2f}"
    ),
    category_orders={
        "Region": selected_regions,
        "Indicator": ["GDP", "Freedom"]
    },
    color_discrete_map=indicator_colours
)

# Improve labels and hover information
fig3.update_traces(
    textposition="outside",
    textfont=dict(size=11, color="#263238"),
    cliponaxis=False,
    marker_line_width=0,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "%{fullData.name}: %{y:.2f}"
        "<extra></extra>"
    )
)

# Professional chart layout
fig3.update_layout(
    title=dict(
        text=(
            "GDP and Freedom Indicators Vary Across Five World Regions"
            "<br><sup>"
            "Regional averages of logged GDP per capita and freedom "
            "| World Happiness Report 2023"
            "</sup>"
        ),
        x=0.02,
        xanchor="left",
        font=dict(size=21, color="#1F2933")
    ),
    xaxis=dict(
        title="World Region",
        tickangle=-15,
        tickfont=dict(size=11),
        showgrid=False,
        automargin=True
    ),
    yaxis=dict(
        title="Average Indicator Value",
        rangemode="tozero",
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor="#52616B",
        showgrid=True,
        gridcolor="#E9EEF2",
        tickfont=dict(size=11),
        automargin=True
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(
        family="Arial, sans-serif",
        color="#263238"
    ),
    margin=dict(l=85, r=55, t=135, b=120),
    height=650,
    bargap=0.28,
    bargroupgap=0.12,
    legend=dict(
        title="Indicator",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    hoverlabel=dict(
        bgcolor="white",
        font_size=12
    )
)

# Display the chart underneath this cell
fig3.show()

Average GDP and Freedom Indicators by Region


Region,GDP,Freedom
Western Europe,10.86,0.85
Latin America,9.29,0.84
East Asia,10.36,0.77
Sub-Saharan Africa,8.07,0.70
South Asia,8.51,0.75
